In [1]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

# path = "/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master"
# grewpy.set_config('ud')
path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_Wolof-WTB"
grewpy.set_config('ud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

connected to port: 64722


In [2]:
all_matches = corpus.search(grewpy.Request("pattern{X[upos<>PUNCT]}").without("X[InIdiom=Yes]").without("X[Idiom=Yes]").without("X[InTitle=Yes]").without("X[Title=Yes]").without("X[Scrap=Yes]").without("X[Foreign]").without("X[Lang]").without("X-[fixed]->Y").without("Y-[flat:name]->X").without("Y-[goeswith]->X") , clustering_parameter=['X.lemma'])

In [3]:
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 10:
        matches[key] = value

In [4]:
print(len(matches))

408


In [5]:
# Create a dictionary to map sent_id to sentences for quick lookup
sent_id_to_sentence = {draft[i].meta['sent_id']: draft[i].features for i in range(len(draft))}

match_upos = {}
for key, value in matches.items():
    for m in value:
        match_sent_id = m['sent_id']
        match_node_index = str(m['matching']['nodes']['X'])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[match_node_index]
                if 'ExtPos' in current_token_features:
                    match_upos.setdefault((key, current_token_features['ExtPos']), []).append(m)
                else:
                    match_upos.setdefault((key, current_token_features['upos']), []).append(m)

In [6]:
new_match_upos = {}
for key, value in match_upos.items():
    if len(value) > 10:
        new_match_upos[key] = value
match_upos = new_match_upos

for key, value in match_upos.items():
    print(key, len(value))

('ñëw', 'VERB') 19
('ñépp', 'PRON') 34
('ñàkk', 'VERB') 16
('ñuul', 'VERB') 15
('ñeent', 'NUM') 29
('ñeel', 'ADP') 21
('ñatt', 'NUM') 18
('ñaar', 'NUM') 59
('ëpp', 'VERB') 43
('ëmb', 'VERB') 12
('àtte', 'NOUN') 23
('àq', 'NOUN') 19
('ànd', 'VERB') 33
('àdduna', 'NOUN') 77
('àddina', 'NOUN') 30
('yàtt', 'NOUN') 12
('yàgg', 'VERB') 32
('yu', 'ADP') 32
('yore', 'VERB') 13
('yor', 'VERB') 36
('yoon', 'NOUN') 71
('yomb', 'VERB') 11
('yokk', 'VERB') 16
('yit', 'ADV') 12
('yem', 'VERB') 17
('yelloo', 'VERB') 12
('yeew', 'NOUN') 11
('yaxantu', 'NOUN') 11
('yaram', 'NOUN') 21
('yaay', 'NOUN') 11
('xool', 'VERB') 12
('xeltu', 'NOUN') 11
('xel', 'NOUN') 36
('xeex', 'VERB') 18
('xeet', 'NOUN') 36
('xayma', 'NOUN') 14
('xaw', 'VERB') 12
('xarnu', 'NOUN') 48
('xarit', 'NOUN') 12
('xare', 'NOUN') 19
('xarala', 'NOUN') 12
('xar', 'NOUN') 29
('xanaa', 'ADV') 14
('xam-xam', 'NOUN') 71
('xam', 'VERB') 165
('xale', 'NOUN') 36
('xalaat', 'NOUN') 30
('xalaat', 'VERB') 26
('xaar', 'VERB') 11
('xaalis', 'NOUN

In [7]:
with open("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

In [8]:
data = { k : list() for k in match_upos }
for node, mts in match_upos.items():
    for match in mts:
        features = grex.data.extract_features(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[node].append(formatted_features)

In [9]:
unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, match_upos in data.items() for m in match_upos for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_lemma) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [11]:
unique_features

['node:X:child:Aspect=Hab',
 'node:X:child:Aspect=Imp',
 'node:X:child:Aspect=Perf',
 'node:X:child:Aspect=Prog',
 'node:X:child:Case=Acc',
 'node:X:child:Case=Gen',
 'node:X:child:Case=Nom',
 'node:X:child:Definite=Def',
 'node:X:child:Definite=Ind',
 'node:X:child:Deixis=Med',
 'node:X:child:Deixis=Prox',
 'node:X:child:Deixis=Remt',
 'node:X:child:DeixisRef=1',
 'node:X:child:DeixisRef=2',
 'node:X:child:FocusType=Compl',
 'node:X:child:FocusType=Subj',
 'node:X:child:FocusType=Verb',
 'node:X:child:Gender=Fem',
 'node:X:child:Gender=Masc',
 'node:X:child:Mood=Cnd',
 'node:X:child:Mood=Imp',
 'node:X:child:Mood=Ind',
 'node:X:child:Mood=Opt',
 'node:X:child:NounClass=Wol1',
 'node:X:child:NounClass=Wol10',
 'node:X:child:NounClass=Wol11',
 'node:X:child:NounClass=Wol12',
 'node:X:child:NounClass=Wol2',
 'node:X:child:NounClass=Wol3',
 'node:X:child:NounClass=Wol4',
 'node:X:child:NounClass=Wol5',
 'node:X:child:NounClass=Wol6',
 'node:X:child:NounClass=Wol7',
 'node:X:child:NounClas

In [10]:
X = np.zeros((len(data.keys()), len(unique_features)))
for adv, samples in data.items():
    n_samples = len(matches)
    for m in samples:
        for feature in m:
            X[adv2idx[adv], feature2idx[feature]] += 1
    X[adv2idx[adv]] = X[adv2idx[adv]] / n_samples
print(f"{X.shape=}")

X.shape=(420, 420)


In [11]:
import numpy as np
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score

def find_optimal_clusters(X, max_clusters=20, metric='cosine', method='complete'):
    distance_matrix = pdist(X, metric=metric)
    linked = linkage(distance_matrix, method=method, optimal_ordering=True)
    
    silhouette_scores = []
    for num_clusters in range(2, max_clusters + 1):
        labels = fcluster(linked, num_clusters, criterion='maxclust')
        if len(np.unique(labels)) > 1:  # Ensure there is more than one cluster
            score = silhouette_score(X, labels, metric=metric)
            silhouette_scores.append(score)
            print(f'Number of clusters: {num_clusters}, Silhouette Score: {score}')
        else:
            silhouette_scores.append(-1)  # Append a low score if only one cluster
    
    optimal_clusters = np.argmax(silhouette_scores) + 2  # +2 because range starts from 2
    return optimal_clusters, silhouette_scores

# Find the optimal number of clusters
optimal_clusters, silhouette_scores = find_optimal_clusters(X, max_clusters=50)
print(f'Optimal number of clusters: {optimal_clusters}')
print(X.shape)

Number of clusters: 2, Silhouette Score: 0.2553419880433947
Number of clusters: 3, Silhouette Score: 0.24830707243086672
Number of clusters: 4, Silhouette Score: 0.25120196311589027
Number of clusters: 5, Silhouette Score: 0.2743106926090152
Number of clusters: 6, Silhouette Score: 0.2807614580753989
Number of clusters: 7, Silhouette Score: 0.2648633743931654
Number of clusters: 8, Silhouette Score: 0.39772850842271046
Number of clusters: 9, Silhouette Score: 0.39549672863428087
Number of clusters: 10, Silhouette Score: 0.39927052013301845
Number of clusters: 11, Silhouette Score: 0.4043118457458254
Number of clusters: 12, Silhouette Score: 0.41192835642397824
Number of clusters: 13, Silhouette Score: 0.3293569143975827
Number of clusters: 14, Silhouette Score: 0.331034613870725
Number of clusters: 15, Silhouette Score: 0.3347651055058589
Number of clusters: 16, Silhouette Score: 0.24451431405076338
Number of clusters: 17, Silhouette Score: 0.2517888740495086
Number of clusters: 18, Si

In [12]:
distance_matrix = pdist(X, metric='cosine')
linked = linkage(distance_matrix, method="complete", optimal_ordering=True)
labels = fcluster(linked, optimal_clusters, criterion='maxclust')
clusters = {i: [] for i in range(1, optimal_clusters + 1)}
for i, label in enumerate(labels):
    clusters[label].append(i)

In [13]:
for cluster, members in clusters.items():
    print(f'Cluster {cluster}:')
    for member in members:
        print(f'  {unique_lemma[member]}')

Cluster 1:
  ('aju', 'VERB')
  ('am', 'VERB')
  ('amal', 'VERB')
  ('baax', 'VERB')
  ('bare', 'VERB')
  ('bari', 'VERB')
  ('bañ', 'VERB')
  ('bees', 'VERB')
  ('bind', 'VERB')
  ('bokk', 'VERB')
  ('boole', 'VERB')
  ('bàyyi', 'VERB')
  ('bëgg', 'VERB')
  ('dal', 'VERB')
  ('def', 'VERB')
  ('defar', 'VERB')
  ('defe', 'VERB')
  ('dellu', 'VERB')
  ('dem', 'VERB')
  ('des', 'VERB')
  ('di', 'VERB')
  ('doon', 'VERB')
  ('door', 'VERB')
  ('dox', 'VERB')
  ('doxal', 'VERB')
  ('doy', 'VERB')
  ('dugg', 'VERB')
  ('dund', 'VERB')
  ('dégg', 'VERB')
  ('dëgër', 'VERB')
  ('dëkk', 'VERB')
  ('fal', 'VERB')
  ('feeñ', 'VERB')
  ('fekk', 'VERB')
  ('gis', 'VERB')
  ('génn', 'VERB')
  ('gën', 'VERB')
  ('indi', 'VERB')
  ('jaar', 'VERB')
  ('jiite', 'VERB')
  ('jiitu', 'VERB')
  ('jot', 'VERB')
  ('jox', 'VERB')
  ('joxe', 'VERB')
  ('jublu', 'VERB')
  ('jur', 'VERB')
  ('jàng', 'VERB')
  ('jàpp', 'VERB')
  ('jéem', 'VERB')
  ('jëfandikoo', 'VERB')
  ('jëkk', 'VERB')
  ('jël', 'VERB')
  ('j

In [14]:
pie_chart = {}
for cluster, members in clusters.items():
    pie_chart[cluster] = {}
    for member in members:
        if unique_lemma[member][1] in pie_chart[cluster]:
            pie_chart[cluster][unique_lemma[member][1]] += 1
        else:
            pie_chart[cluster][unique_lemma[member][1]] = 1

In [15]:
pie_chart

{1: {'VERB': 140},
 2: {'PROPN': 20, 'NOUN': 118, 'PRON': 1},
 3: {'NOUN': 3},
 4: {'NOUN': 3, 'ADP': 1},
 5: {'PROPN': 10},
 6: {'ADV': 25},
 7: {'CCONJ': 10, 'AUX': 5, 'PART': 1, 'ADP': 3, 'SCONJ': 7, 'PRON': 1},
 8: {'PROPN': 8, 'NOUN': 2, 'DET': 1},
 9: {'PROPN': 5, 'PRON': 13, 'NOUN': 6, 'AUX': 3, 'ADV': 3, 'VERB': 1},
 10: {'ADP': 12},
 11: {'NUM': 8},
 12: {'DET': 10}}

In [16]:
# make pie chart with plotly
import plotly.express as px
import plotly.graph_objects as go

# Function to create pie chart for a given cluster
def create_pie_chart(cluster_number):
    labels = [f'{k} ({v})' for k, v in pie_chart[cluster_number].items()]
    values = list(pie_chart[cluster_number].values())
    fig = go.Figure(data=[go.Pie(labels=labels, values=values)])
    return fig

# Create initial pie chart
fig = create_pie_chart(1)

# Add dropdown menu
dropdown_buttons = [
    {
        'label': f'Cluster {i}',
        'method': 'update',
        'args': [{'values': [list(pie_chart[i].values())], 'labels': [[f'{k} ({v})' for k, v in pie_chart[i].items()]]}]
    } for i in pie_chart.keys()
]

fig.update_layout(
    updatemenus=[
        {
            'buttons': dropdown_buttons,
            'direction': 'down',
            'showactive': True,
        }
    ]
)

fig.show()

In [22]:
fig.write_html("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/output/visualisations/Wolof_all_lex_units_pie.html")

In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Dynamically generate a color palette for all unique categories
all_categories = set(cat for cluster in pie_chart.values() for cat in cluster.keys())
color_palette = px.colors.qualitative.Plotly  # Use Plotly's default qualitative color palette
category_colors = {cat: color_palette[i % len(color_palette)] for i, cat in enumerate(sorted(all_categories))}

# Determine the number of rows and columns for the subplots
num_clusters = len(pie_chart)
cols = 4  # Fixed number of columns
rows = -(-num_clusters // cols)  # Calculate rows dynamically (ceiling division)

# Create a subplot layout
fig = make_subplots(rows=rows, cols=cols, specs=[[{'type': 'domain'}]*cols]*rows)

# Add each pie chart to the subplot
for i, (cluster_number, cluster_data) in enumerate(pie_chart.items(), start=1):
    labels = list(cluster_data.keys())
    values = list(cluster_data.values())
    colors = [category_colors[label] for label in labels]
    
    fig.add_trace(
        go.Pie(labels=labels, values=values, marker=dict(colors=colors), name=f'Cluster {cluster_number}'),
        row=(i-1)//cols + 1, col=(i-1)%cols + 1
    )

# Add a dummy pie chart to create a legend
legend_labels = list(category_colors.keys())
legend_values = [1] * len(legend_labels)  # Dummy values for the legend
legend_colors = [category_colors[label] for label in legend_labels]

fig.add_trace(
    go.Pie(
        labels=legend_labels,
        values=legend_values,
        marker=dict(colors=legend_colors),
        name="Legend",
        showlegend=True,
        hoverinfo="label"  # Only show labels on hover
    ),
    row=1, col=cols  # Place the legend in the first row, last column
)

# Update layout
fig.update_layout(
    title_text="Dynamic Pie Charts for Each Cluster",
    height=300 * rows,  # Adjust height dynamically based on rows
    width=1200,  # Fixed width
    showlegend=True,
    legend=dict(
        x=1.05,  # Position the legend to the right of the chart
        y=0.5,
        traceorder="normal"
    )
)

# Show the figure
fig.show()

In [18]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

In [23]:


# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_tsne[:, 0],
    'PCA2': X_tsne[:, 1],
    'Cluster': labels,
    'Word': unique_lemma
})
# pca1_min, pca1_max = df['PCA1'].min(), df['PCA1'].max()
# pca2_min, pca2_max = df['PCA2'].min(), df['PCA2'].max()
# Create a scatter plot with Plotly
fig = go.Figure()

# Add traces for each cluster
for cluster in range(1, optimal_clusters + 1):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        hovertemplate='%{text}<extra></extra>',
    ))

# Update layout with dropdown menu
fig.update_layout(
    title='Word Clusters',
    xaxis_title='tsne1',
    yaxis_title='tsne2',
    width=1600,  # Set the width of the figure
    height=800,  # Set the height of the figure
    # xaxis=dict(range=[pca1_min, pca1_max]),
    # yaxis=dict(range=[pca2_min, pca2_max]),
    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * optimal_clusters},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(optimal_clusters)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, optimal_clusters + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

In [24]:
fig.write_html("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/output/visualisations/Wolof_all_lex_units_tsne.html")